<a href="https://colab.research.google.com/github/Maverick-Ansh/rlt-reproduce/blob/master/rltpaper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch, subprocess, sys
print(sys.version)
print("torch", torch.__version__, "cuda", torch.version.cuda, "avail", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory/1e9:.1f} GB", "sm", f"{p.major}.{p.minor}")
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.used","--format=csv"],capture_output=True,text=True).stdout)
print(subprocess.run(["df","-h","/content"],capture_output=True,text=True).stdout)




"just the code is there for that m"

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
torch 2.11.0+cu128 cuda 12.8 avail True
Tesla T4 15.6 GB sm 7.5
name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 3 MiB

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  43% /



In [ ]:
# Keep-alive + hardware baseline for the RLT reproduction.
# The recurrent decoder is a per-token Python loop over small matmuls, so what
# actually matters on a T4 is kernel-launch latency, not peak FLOPs. Measure it.
import torch, time, os
torch.backends.cuda.matmul.allow_tf32 = False   # T4 has no TF32; be explicit
dev = "cuda"
os.makedirs("/content/work", exist_ok=True)

def bench(fn, n=50, warmup=10):
    for _ in range(warmup): fn()
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return (time.time() - t0) / n * 1e3

d, B = 256, 256
x = torch.randn(B, d, device=dev); w = torch.randn(d, d, device=dev)
print(f"single [256,256]x[256,256] matmul : {bench(lambda: x @ w):.4f} ms  <- launch-bound")
big = torch.randn(4096, 4096, device=dev)
print(f"[4096,4096]^2 matmul fp32         : {bench(lambda: big @ big, n=10):.2f} ms")
bigh = big.half()
print(f"[4096,4096]^2 matmul fp16         : {bench(lambda: bigh @ bigh, n=10):.2f} ms")
print()
# 32 sequential steps x 4 layers x ~8 small ops = the shape of one RLT forward
def fake_recurrence():
    h = x
    for _ in range(32 * 4 * 8): h = h @ w
print(f"1024 chained small matmuls        : {bench(fake_recurrence, n=5):.1f} ms  <- per-step budget")
print("\ntorch", torch.__version__, "| free GB", torch.cuda.mem_get_info()[0]/1e9)

single [256,256]x[256,256] matmul : 0.0426 ms  <- launch-bound
[4096,4096]^2 matmul fp32         : 31.16 ms
[4096,4096]^2 matmul fp16         : 5.53 ms

1024 chained small matmuls        : 19.9 ms  <- per-step budget

torch 2.11.0+cu128 | free GB 15.340470272


In [ ]:
import subprocess, os, sys
REPO = "https://github.com/Maverick-Ansh/rlt-reproduce.git"
if not os.path.isdir("/content/rlt-reproduce"):
    print(subprocess.run(["git","clone","-q",REPO,"/content/rlt-reproduce"],capture_output=True,text=True))
os.chdir("/content/rlt-reproduce")
print(subprocess.run(["git","pull","-q","--ff-only"],capture_output=True,text=True).stderr[-500:])
print(subprocess.run(["git","log","--oneline","-1"],capture_output=True,text=True).stdout)
print(subprocess.run([sys.executable,"check_task.py"],capture_output=True,text=True).stdout)










CompletedProcess(args=['git', 'clone', '-q', 'https://github.com/Maverick-Ansh/rlt-reproduce.git', '/content/rlt-reproduce'], returncode=0, stdout='', stderr='')

6e6f587 RLT: model, task, exactness checks

EVALUATION BRACKET
  floor   (uniform guessing) : 0.0167
  ceiling (exact ground truth): 1.0000
  the metric is direct accuracy; no learned probe sits between the
  model and the number, so the bracket cannot narrow during training.

--- A5 ---
  order                       : 60
  identity                    : ok
  inverses / Latin square     : ok
  associativity (20k triples) : ok
  abelian                     : False   <- NC1-complete word problem
  label marginal: min 0.0162  max 0.0170 (uniform = 0.0167)
  best constant-output policy : 0.0170   (chance = 0.0167)
  oracle shortcut policies (fit on the eval data itself):
    last 1 input token(s)        : 0.0322
    last 2 input token(s)        : 0.0477
    last 3 input token(s)        : 0.4038
    position index only          : 0

In [ ]:
import subprocess, sys, os
os.chdir("/content/rlt-reproduce")
def sh(*a, timeout=1200):
    r = subprocess.run(list(a), capture_output=True, text=True, timeout=timeout)
    print(r.stdout[-8000:])
    if r.returncode: print("STDERR:", r.stderr[-4000:])
    return r.returncode
sh("git","pull","-q","--ff-only")
sh(sys.executable,"check_task.py")


EVALUATION BRACKET
  floor   (uniform guessing) : 0.0167
  ceiling (exact ground truth): 1.0000
  the metric is direct accuracy; no learned probe sits between the
  model and the number, so the bracket cannot narrow during training.

--- A5 ---
  order                       : 60
  identity                    : ok
  inverses / Latin square     : ok
  associativity (20k triples) : ok
  abelian                     : False   <- NC1-complete word problem
  label marginal: min 0.0162  max 0.0170 (uniform = 0.0167)
  best constant-output policy : 0.0170   (chance = 0.0167)
  oracle shortcut policies (fit on half, scored on held-out half):
    last 1 input token(s)        : 0.0319
    last 2 input token(s)        : 0.0185
    last 3 input token(s)        : 0.0168
    position index only          : 0.0165

--- Z60 ---
  order                       : 60
  identity                    : ok
  inverses / Latin square     : ok
  associativity (20k triples) : ok
  abelian                     : True  

0

In [ ]:
sh(sys.executable,"smoke.py","--device","cuda")

device=cuda dtype=torch.float32  model: d=64 L_E=3 L_D=3 W=4 tied=True  params=177,024
measured numerical noise floor (same computation, different batch padding): 2.384e-07

C1  Prop. 3.1 -- invariance to the serving split
  parallel prefill  vs  fully incremental : 4.023e-07
  worst over all 24 split points T     : 5.253e-07  (at T=4)
  measured numerical noise floor          : 2.384e-07
  VERDICT: CONFIRMED (split difference is 2.2x the noise floor)

C2  Prop. B.1 -- causality of the complete state H_t
  max |s_j(x) - s_j(x')| for j < t (perturbed tail) : 0.000e+00
  max |logit_j difference| for j < t                : 0.000e+00
  max |logit_j difference| for j >= t  (sanity)     : 6.886e-01
  VERDICT: CONFIRMED (bitwise)

C3  Sec. 2.4 -- a token-parallel SWA decoder pass is not the recurrence

  alpha = 1.0   (alpha = 0 severs Eq. (2.11)'s feedback term)
    1 parallel sweep(s): positions exact =   1   max err overall = 7.824e-01
    2 parallel sweep(s): positions exact =   2   max e

0

In [ ]:
sh("git","pull","-q","--ff-only")
# 40-step smoke of the real training path: check step time and that loss moves.
sh(sys.executable,"sweep.py","--steps","40","--seeds","0","--out","results_smoke",
   "--logs","logs_smoke","--groups","a5")
import glob, json
for f in sorted(glob.glob("results_smoke/*.json")):
    d = json.load(open(f))
    per = d["wall_s"]/d["args"]["steps"]
    print(f"{d['tag']:<22s} params {d['params']:>9,}  {per*1000:6.0f} ms/step  "
          f"-> {per*2500/60:5.1f} min for 2500 steps")


4 runs: groups=['a5'] arms=['rlt', 'rlt_a0', 'rlt_untied', 'plain'] seeds=[0]
[1/4] a5_rlt_s0  (0s elapsed)
[2/4] a5_rlt_a0_s0  (85s elapsed)
[3/4] a5_rlt_untied_s0  (164s elapsed)
[4/4] a5_plain_s0  (243s elapsed)
sweep done in 259s

a5_plain_s0            params 6,326,016     309 ms/step  ->  12.9 min for 2500 steps
a5_rlt_a0_s0           params 3,902,976    1892 ms/step  ->  78.8 min for 2500 steps
a5_rlt_s0              params 3,902,976    1916 ms/step  ->  79.8 min for 2500 steps
a5_rlt_untied_s0       params 7,048,704    1862 ms/step  ->  77.6 min for 2500 steps


In [ ]:
for t in ["a5_rlt_s0","a5_plain_s0"]:
    print("="*60, t)
    print(open(f"logs_smoke/{t}.log").read())

============================================================ a5_rlt_s0
[a5_rlt_s0] params=3,902,976 device=cuda blocks/token=6 chance=0.0167
  step     0  loss 4.1507  acc 0.0198  |g| 1.20  2s
  step    39  loss 4.0968  acc 0.0175  |g| 0.25  62s
  eval L=  16  acc 0.0223  final-position acc 0.0127
  eval L=  32  acc 0.0186  final-position acc 0.0146
  eval L=  64  acc 0.0168  final-position acc 0.0176
  eval L= 128  acc 0.0172  final-position acc 0.0146
  eval L= 256  acc 0.0169  final-position acc 0.0146
[a5_rlt_s0] wrote results_smoke/a5_rlt_s0.json  (77s)

============================================================ a5_plain_s0
[a5_plain_s0] params=6,326,016 device=cuda blocks/token=6 chance=0.0167
  step     0  loss 4.1723  acc 0.0164  |g| 1.34  1s
  step    39  loss 4.0957  acc 0.0175  |g| 0.23  9s
  eval L=  16  acc 0.0228  final-position acc 0.0117
  eval L=  32  acc 0.0182  final-position acc 0.0137
  eval L=  64  acc 0.0172  final-position acc 0.0176
  eval L= 128  acc 0.0178 

In [ ]:
sh("git","pull","-q","--ff-only")
sh(sys.executable,"smoke_static.py","--length","32","--batch","192","--compile")


C7  Sec. 4.2 -- the fixed-slot kernel is the same model
  max |logit_ref - logit_fast|         : 7.153e-07   (relative 5.26e-07)
  max |grad_ref - grad_fast|           : 2.468e-08
  cosine(grad_ref, grad_fast)          : 1.0000000000
  VERDICT: CONFIRMED

Speed (this is an implementation number, NOT a claim about the paper's
hardware co-design -- Sec. 4 makes no measured speedup claim and neither
does this repository.)
  reference growing-cache recurrence :    742.8 ms / fwd+bwd
  fixed-slot recurrence              :    703.5 ms (1.06x)
  fixed-slot + torch.compile         :    320.9 ms (2.31x)   [compile+warmup included above]



0

In [ ]:
import torch, time, torch.nn.functional as F
sys.path.insert(0,"/content/rlt-reproduce")
from rlt import RLTConfig, RLT
from rlt.tasks import GroupTask, VOCAB

def timeit(batch, length, layers=3, d=256, mode=None, n=6):
    cfg = RLTConfig(vocab_size=VOCAB, d_model=d, n_heads=4, L_E=layers, L_D=layers,
                    d_ff=4*d, W=8, alpha=1.0, tied=True, max_position=1024)
    torch.manual_seed(0); m = RLT(cfg).cuda(); m.use_static = True
    if mode: m.enable_compile(mode=mode)
    task = GroupTask("a5", device="cuda")
    x, y = task.sample(batch, length)
    def one():
        logits, _ = m(x)
        F.cross_entropy(logits[:,1:].reshape(-1,VOCAB), y.reshape(-1)).backward()
        m.zero_grad(set_to_none=True)
    t0=time.time()
    for _ in range(3): one()
    torch.cuda.synchronize(); warm=time.time()-t0
    t0=time.time()
    for _ in range(n): one()
    torch.cuda.synchronize()
    del m; torch.cuda.empty_cache()
    return (time.time()-t0)/n*1000, warm

for batch in (192, 512, 1024):
    ms,_ = timeit(batch, 32, mode="default")
    print(f"batch {batch:5d} len 32 compile=default : {ms:7.1f} ms/step  "
          f"({ms/batch*1000:.3f} ms per 1k seqs)")
print()
ms, warm = timeit(512, 32, mode="reduce-overhead")
print(f"batch  512 len 32 reduce-overhead : {ms:7.1f} ms/step  (warmup {warm:.0f}s)")
ms, _ = timeit(512, 32, layers=2, mode="default")
print(f"batch  512 len 32 layers=2        : {ms:7.1f} ms/step")

W0913 05:23:41.865000 4296 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


batch   192 len 32 compile=default :   354.2 ms/step  (1844.646 ms per 1k seqs)
batch   512 len 32 compile=default :   487.1 ms/step  (951.395 ms per 1k seqs)
batch  1024 len 32 compile=default :   906.7 ms/step  (885.449 ms per 1k seqs)



/usr/local/lib/python3.13/dist-packages/torch/_inductor/cudagraph_trees.py:2582: UserWarning: Unable to hit fast path of CUDAGraphs because of pending, uninvoked backwards. Consider running with torch.no_grad() or using torch.compiler.cudagraph_mark_step_begin() before each model invocation
  warnings.warn(
W0913 05:26:28.093000 4296 torch/_dynamo/convert_frame.py:1743] [0/8] torch._dynamo hit config.recompile_limit (8)
W0913 05:26:28.093000 4296 torch/_dynamo/convert_frame.py:1743] [0/8]    function: '_step_static' (/content/rlt-reproduce/rlt/model.py:431)
W0913 05:26:28.093000 4296 torch/_dynamo/convert_frame.py:1743] [0/8]    last reason: 0/7: len(ck) == 3                                             # z, a, b = layer.forward_step_static(z, pos, ck[l], cv[l], hist_mask,  # ontent/rlt-reproduce/rlt/model.py:436 in _step_static
W0913 05:26:28.093000 4296 torch/_dynamo/convert_frame.py:1743] [0/8] User stack trace:
W0913 05:26:28.093000 4296 torch/_dynamo/convert_frame.py:1743] [0/8]   

batch  512 len 32 reduce-overhead :   474.3 ms/step  (warmup 52s)
batch  512 len 32 layers=2        :   552.8 ms/step


In [ ]:
import subprocess, sys, os, time
os.chdir("/content/rlt-reproduce")
subprocess.run(["git","pull","-q","--ff-only"])

# layers=2 -> 4 blocks/token. log2(32) = 5 > 4, so a fixed-depth model provably
# cannot run a full parallel scan over a length-32 word: E1 becomes falsifiable
# in-distribution, not only under extrapolation.
COMMON = ["--steps","1600","--length","32","--batch","384","--layers","2",
          "--d-model","256","--groups","a5,z60","--arms","rlt,rlt_a0,plain",
          "--seeds","0,1"]
procs = []
for i in range(2):
    log = open(f"/content/work/sweep{i}.log","w")
    p = subprocess.Popen([sys.executable,"sweep.py",*COMMON,"--shard",f"{i}/2"],
                         stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    procs.append(p); print("launched shard", i, "pid", p.pid)
time.sleep(20)
for i in range(2): print(open(f"/content/work/sweep{i}.log").read())

launched shard 0 pid 12772
launched shard 1 pid 12773
6 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/6] a5_rlt_s0  (0s elapsed)

6 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/6] a5_rlt_s1  (0s elapsed)



In [ ]:
import glob, time, subprocess
def status():
    print(subprocess.run(["bash","-lc","nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader"],
                         capture_output=True,text=True).stdout.strip())
    for i in range(2):
        print(f"--- shard {i} ---"); print(open(f"/content/work/sweep{i}.log").read().strip()[-400:])
    for f in sorted(glob.glob("logs/*.log")):
        tail = [l for l in open(f).read().strip().split("\n") if l.strip()][-2:]
        print(f"  {f}: " + " | ".join(t.strip() for t in tail))
status()

0 %, 10047 MiB
--- shard 0 ---
6 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/6] a5_rlt_s0  (0s elapsed)
--- shard 1 ---
6 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/6] a5_rlt_s1  (0s elapsed)
  logs/a5_rlt_s0.log: W0913 05:27:59.593000 12774 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
  logs/a5_rlt_s1.log: W0913 05:27:59.631000 12775 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


In [ ]:
print(subprocess.run(["bash","-lc","nproc; uptime; ps -eo pid,etimes,pcpu,rss,comm --sort=-pcpu | head -12"],
                     capture_output=True,text=True).stdout)

2
 05:28:58 up 51 min,  0 user,  load average: 3.30, 1.75, 1.01
    PID ELAPSED %CPU   RSS COMMAND
  12774      82 55.9 1571952 python3
  12775      82 55.7 1570876 python3
  12904      60 12.3 495480 python3
  12892      61 11.9 495572 python3
  12890      61 11.0 495976 python3
  12902      60 11.0 495328 python3
   4296    2019  8.4 1889460 python3
  12815      74  8.0 671000 python3
  12814      74  7.9 671016 python3
  11545     326  1.8 673280 python3
   7178    1339  1.5 315788 node



In [ ]:
import signal, os
for p in procs:
    try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
    except Exception as e: print("kill", e)
time.sleep(3)
print(subprocess.run(["bash","-lc","ps -eo pid,comm | grep -c python3"],capture_output=True,text=True).stdout)
import shutil
for d in ("results","logs"): shutil.rmtree(d, ignore_errors=True)
print("cleared partial results")

5

cleared partial results


In [ ]:
subprocess.run(["git","pull","-q","--ff-only"])
# One worker: 2 vCPUs and a Python-bound decoder loop mean concurrency buys nothing.
COMMON = ["--steps","1200","--length","32","--batch","384","--layers","2",
          "--d-model","256","--groups","a5,z60","--arms","rlt,rlt_a0,plain",
          "--seeds","0,1"]
log = open("/content/work/sweep.log","w")
sweep = subprocess.Popen([sys.executable,"sweep.py",*COMMON],
                         stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("launched pid", sweep.pid)
time.sleep(90)
print(open("/content/work/sweep.log").read())
print(subprocess.run(["bash","-lc","tail -3 logs/*.log 2>/dev/null"],capture_output=True,text=True).stdout)

launched pid 13599
12 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/12] a5_rlt_s0  (0s elapsed)

  step   100  loss 3.9741  acc 0.0486  |g| 0.24  33s
  step   200  loss 3.9455  acc 0.0538  |g| 0.26  60s
  step   300  loss 3.8428  acc 0.0781  |g| 0.26  86s



In [ ]:
print(open("/content/work/sweep.log").read()[-600:])
print(subprocess.run(["bash","-lc","for f in logs/*.log; do echo \"== $f\"; grep -E 'step|eval' $f | tail -6; done"],
                     capture_output=True,text=True).stdout)

12 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/12] a5_rlt_s0  (0s elapsed)

== logs/a5_rlt_s0.log
  step     0  loss 4.1592  acc 0.0153  |g| 1.03  6s
  step   100  loss 3.9741  acc 0.0486  |g| 0.24  33s
  step   200  loss 3.9455  acc 0.0538  |g| 0.26  60s
  step   300  loss 3.8428  acc 0.0781  |g| 0.26  86s



In [ ]:
time.sleep(240)   # bounded wait; never an unbounded poll loop in an MCP cell
print(open("/content/work/sweep.log").read()[-800:])
print(subprocess.run(["bash","-lc","for f in logs/*.log; do echo \"== $f\"; grep -E 'step |eval ' $f | tail -4; done"],
                     capture_output=True,text=True).stdout)

12 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/12] a5_rlt_s0  (0s elapsed)
[2/12] a5_rlt_s1  (334s elapsed)

== logs/a5_rlt_s0.log
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 0.8841  final-position acc 0.3379
  eval L= 128  acc 0.5172  final-position acc 0.0371
  eval L= 256  acc 0.2791  final-position acc 0.0195
== logs/a5_rlt_s1.log
  step     0  loss 4.1634  acc 0.0160  |g| 0.98  5s



In [ ]:
# NOTE: deliberately NOT pulling -- sweep.py spawns a fresh train.py per job, so a
# mid-sweep pull would run later jobs on different code than earlier ones.
time.sleep(270)
print(open("/content/work/sweep.log").read()[-700:])
print(subprocess.run(["bash","-lc","for f in logs/*.log; do echo \"== $f\"; grep -E 'eval |step  *1100' $f | tail -5; done"],
                     capture_output=True,text=True).stdout)

12 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/12] a5_rlt_s0  (0s elapsed)
[2/12] a5_rlt_s1  (334s elapsed)
[3/12] a5_rlt_a0_s0  (667s elapsed)

== logs/a5_rlt_a0_s0.log
== logs/a5_rlt_s0.log
  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 0.8841  final-position acc 0.3379
  eval L= 128  acc 0.5172  final-position acc 0.0371
  eval L= 256  acc 0.2791  final-position acc 0.0195
== logs/a5_rlt_s1.log
  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 1.0000  final-position acc 1.0000
  eval L= 128  acc 0.9969  final-position acc 0.9746
  eval L= 256  acc 0.8826  final-position acc 0.5410



In [ ]:
time.sleep(280)
print(open("/content/work/sweep.log").read()[-700:])
print(subprocess.run(["bash","-lc","for f in logs/*.log; do echo \"== $f\"; grep -E 'eval ' $f | tail -5; done"],
                     capture_output=True,text=True).stdout)

12 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/12] a5_rlt_s0  (0s elapsed)
[2/12] a5_rlt_s1  (334s elapsed)
[3/12] a5_rlt_a0_s0  (667s elapsed)

== logs/a5_rlt_a0_s0.log
== logs/a5_rlt_s0.log
  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 0.8841  final-position acc 0.3379
  eval L= 128  acc 0.5172  final-position acc 0.0371
  eval L= 256  acc 0.2791  final-position acc 0.0195
== logs/a5_rlt_s1.log
  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 1.0000  final-position acc 1.0000
  eval L= 128  acc 0.9969  final-position acc 0.9746
  eval L= 256  acc 0.8826  final-position acc 0.5410



In [ ]:
print(subprocess.run(["bash","-lc",
  "date; echo '--- sweep ---'; tail -5 /content/work/sweep.log; "
  "echo '--- a0 log ---'; tail -6 logs/a5_rlt_a0_s0.log; "
  "echo '--- results ---'; ls -la results/ 2>/dev/null; "
  "echo '--- gpu ---'; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader"],
  capture_output=True,text=True).stdout)

Sun Sep 13 05:52:02 AM UTC 2026
--- sweep ---
12 runs: groups=['a5', 'z60'] arms=['rlt', 'rlt_a0', 'plain'] seeds=[0, 1]
[1/12] a5_rlt_s0  (0s elapsed)
[2/12] a5_rlt_s1  (334s elapsed)
[3/12] a5_rlt_a0_s0  (667s elapsed)
[4/12] a5_rlt_a0_s1  (1020s elapsed)
--- a0 log ---
  eval L=  16  acc 0.4210  final-position acc 0.0117
  eval L=  32  acc 0.2120  final-position acc 0.0137
  eval L=  64  acc 0.1181  final-position acc 0.0254
  eval L= 128  acc 0.0669  final-position acc 0.0234
  eval L= 256  acc 0.0410  final-position acc 0.0117
[a5_rlt_a0_s0] wrote results/a5_rlt_a0_s0.json  (346s)
--- results ---
total 20
drwxr-xr-x 2 root root 4096 Sep 13 05:46 .
drwxr-xr-x 8 root root 4096 Sep 13 05:29 ..
-rw-r--r-- 1 root root 3173 Sep 13 05:46 a5_rlt_a0_s0.json
-rw-r--r-- 1 root root 3125 Sep 13 05:35 a5_rlt_s0.json
-rw-r--r-- 1 root root 3112 Sep 13 05:40 a5_rlt_s1.json
--- gpu ---
95 %, 7297 MiB



In [ ]:
time.sleep(290)
print(subprocess.run(["bash","-lc","tail -4 /content/work/sweep.log; echo; ls results/ | tr '\\n' ' '"],
                     capture_output=True,text=True).stdout)

[1/12] a5_rlt_s0  (0s elapsed)
[2/12] a5_rlt_s1  (334s elapsed)
[3/12] a5_rlt_a0_s0  (667s elapsed)
[4/12] a5_rlt_a0_s1  (1020s elapsed)

a5_rlt_a0_s0.json a5_rlt_s0.json a5_rlt_s1.json 


In [ ]:
time.sleep(300)
print(subprocess.run(["bash","-lc","date; tail -6 /content/work/sweep.log; echo; ls results/ | tr '\\n' ' '; echo; "
                      "for f in logs/z60*.log; do echo \"== $f\"; grep 'eval ' $f | tail -5; done 2>/dev/null"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 05:57:39 AM UTC 2026
[1/12] a5_rlt_s0  (0s elapsed)
[2/12] a5_rlt_s1  (334s elapsed)
[3/12] a5_rlt_a0_s0  (667s elapsed)
[4/12] a5_rlt_a0_s1  (1020s elapsed)
[5/12] a5_plain_s0  (1354s elapsed)
[6/12] a5_plain_s1  (1530s elapsed)

a5_plain_s0.json a5_rlt_a0_s0.json a5_rlt_a0_s1.json a5_rlt_s0.json a5_rlt_s1.json 
== logs/z60*.log



In [ ]:
print(subprocess.run(["bash","-lc","for f in logs/a5_plain_s0.log logs/a5_rlt_a0_s1.log; do echo \"== $f\"; grep 'eval ' $f; done"],
                     capture_output=True,text=True).stdout)
time.sleep(260)
print(subprocess.run(["bash","-lc","date; tail -4 /content/work/sweep.log"],capture_output=True,text=True).stdout)

== logs/a5_plain_s0.log
  eval L=  16  acc 0.1426  final-position acc 0.0117
  eval L=  32  acc 0.0802  final-position acc 0.0156
  eval L=  64  acc 0.0489  final-position acc 0.0098
  eval L= 128  acc 0.0316  final-position acc 0.0195
  eval L= 256  acc 0.0247  final-position acc 0.0117
== logs/a5_rlt_a0_s1.log
  eval L=  16  acc 0.1685  final-position acc 0.0176
  eval L=  32  acc 0.0928  final-position acc 0.0078
  eval L=  64  acc 0.0543  final-position acc 0.0117
  eval L= 128  acc 0.0352  final-position acc 0.0215
  eval L= 256  acc 0.0266  final-position acc 0.0117

Sun Sep 13 06:02:10 AM UTC 2026
[4/12] a5_rlt_a0_s1  (1020s elapsed)
[5/12] a5_plain_s0  (1354s elapsed)
[6/12] a5_plain_s1  (1530s elapsed)
[7/12] z60_rlt_s0  (1705s elapsed)



In [ ]:
script = r'''#!/bin/bash
set -x
cd /content/rlt-reproduce
# wait for the main sweep to finish before pulling: sweep.py spawns a fresh
# train.py per job, so pulling mid-sweep would change code between jobs.
while kill -0 %(PID)d 2>/dev/null; do sleep 10; done
git pull -q --ff-only
P=%(PY)s
# extra seeds on the arm whose extrapolation varied wildly between seeds
$P sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 \
   --groups a5 --arms rlt --seeds 2,3
# NoPE: does the length decay come from the recurrent state or from RoPE?
$P sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 \
   --groups a5 --arms rlt_nope --seeds 0,1,2
# E4: untied encoder/decoder
$P sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 \
   --groups a5 --arms rlt_untied --seeds 0
# C6 replay checks
$P rl.py --rl-steps 0 --out results/c6_replay.json
echo FOLLOWUP_DONE
''' % {"PID": sweep.pid, "PY": sys.executable}
open("/content/work/followup.sh","w").write(script)
log = open("/content/work/followup.log","w")
fu = subprocess.Popen(["bash","/content/work/followup.sh"], stdout=log,
                      stderr=subprocess.STDOUT, start_new_session=True)
print("chained follow-up pid", fu.pid, "waiting on sweep pid", sweep.pid)

chained follow-up pid 21874 waiting on sweep pid 13599


In [ ]:
time.sleep(300)
print(subprocess.run(["bash","-lc","date; tail -4 /content/work/sweep.log; echo; "
                      "for f in logs/z60*.log; do echo \"== $f\"; grep 'eval ' $f | head -3; done 2>/dev/null"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 06:08:27 AM UTC 2026
[5/12] a5_plain_s0  (1354s elapsed)
[6/12] a5_plain_s1  (1530s elapsed)
[7/12] z60_rlt_s0  (1705s elapsed)
[8/12] z60_rlt_s1  (2040s elapsed)

== logs/z60_rlt_s0.log
  eval L=  16  acc 0.9994  final-position acc 1.0000
  eval L=  32  acc 0.9957  final-position acc 0.9805
  eval L=  64  acc 0.5392  final-position acc 0.0117
== logs/z60_rlt_s1.log



In [ ]:
time.sleep(300)
print(subprocess.run(["bash","-lc","date; tail -3 /content/work/sweep.log; echo '--- followup ---'; tail -3 /content/work/followup.log 2>/dev/null; echo; "
                      "for f in logs/z60_rlt_a0*.log logs/z60_plain*.log; do echo \"== $f\"; grep 'eval ' $f | head -3; done 2>/dev/null"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 06:19:11 AM UTC 2026
[8/12] z60_rlt_s1  (2040s elapsed)
[9/12] z60_rlt_a0_s0  (2375s elapsed)
[10/12] z60_rlt_a0_s1  (2709s elapsed)
--- followup ---
+ sleep 10
+ kill -0 13599
+ sleep 10

== logs/z60_rlt_a0_s0.log
  eval L=  16  acc 0.9996  final-position acc 0.9961
  eval L=  32  acc 0.9900  final-position acc 0.9551
  eval L=  64  acc 0.5323  final-position acc 0.0254
== logs/z60_rlt_a0_s1.log
== logs/z60_plain*.log



In [ ]:
time.sleep(300)
print(subprocess.run(["bash","-lc","date; tail -3 /content/work/sweep.log; echo '--- followup ---'; grep -vE '^\\+ (sleep|kill)' /content/work/followup.log | tail -6"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 06:24:28 AM UTC 2026
[10/12] z60_rlt_a0_s1  (2709s elapsed)
[11/12] z60_plain_s0  (3044s elapsed)
[12/12] z60_plain_s1  (3220s elapsed)
--- followup ---
+ cd /content/rlt-reproduce



In [ ]:
time.sleep(150)
sh(sys.executable,"analyze.py")

Accuracy on prefix products.  chance = 0.0167, ceiling = 1.0000, trained at length 32
mean +/- std over seeds; * marks the training length

--- A5  (non-abelian, NC1-complete word problem) ---
  arm                              seeds        L=16         L=32*        L=64        L=128        L=256 
  RLT (alpha=1, tied)                  2  1.000+-0.000  1.000+-0.000  0.942+-0.082  0.757+-0.339  0.581+-0.427
  RLT alpha=0  [no recurrence]         2  0.295+-0.179  0.152+-0.084  0.086+-0.045  0.051+-0.022  0.034+-0.010
  plain causal Transformer             2  0.183+-0.058  0.101+-0.029  0.059+-0.014  0.037+-0.008  0.027+-0.004
  recurrence effect at L=32: +0.8476  (seed spread 0.0842) -> above seed noise

--- Z60 (abelian, shallow-computable) ---
  arm                              seeds        L=16         L=32*        L=64        L=128        L=256 
  RLT (alpha=1, tied)                  2  0.999+-0.000  0.990+-0.008  0.531+-0.011  0.275+-0.006  0.145+-0.002
  RLT alpha=0  [no recurrence

0

In [ ]:
time.sleep(290)
print(subprocess.run(["bash","-lc","date; grep -vE '^\\+ (sleep|kill)' /content/work/followup.log | tail -8; echo; ls results/ | tr '\\n' ' '"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 06:32:26 AM UTC 2026
+ cd /content/rlt-reproduce

a5_plain_s0.json a5_plain_s1.json a5_rlt_a0_s0.json a5_rlt_a0_s1.json a5_rlt_s0.json a5_rlt_s1.json z60_plain_s0.json z60_plain_s1.json z60_rlt_a0_s0.json z60_rlt_a0_s1.json z60_rlt_s0.json z60_rlt_s1.json 


In [ ]:
print(subprocess.run(["bash","-lc",
  "date; tail -2 /content/work/sweep.log; ps -p 13599 -o pid,etimes,comm= 2>/dev/null || echo 'sweep pid gone'; "
  "echo '--- followup tail ---'; tail -12 /content/work/followup.log"],
  capture_output=True,text=True).stdout)

Sun Sep 13 06:49:12 AM UTC 2026
[12/12] z60_plain_s1  (3220s elapsed)
sweep done in 3396s
    PID ELAPSED 
sweep pid gone
--- followup tail ---
+ sleep 10
+ kill -0 13599
+ git pull -q --ff-only
+ P=/usr/bin/python3
+ /usr/bin/python3 sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 --groups a5 --arms rlt --seeds 2,3
2 runs: groups=['a5'] arms=['rlt'] seeds=[2, 3]
[1/2] a5_rlt_s2  (0s elapsed)
[2/2] a5_rlt_s3  (335s elapsed)
sweep done in 670s
+ /usr/bin/python3 sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 --groups a5 --arms rlt_nope --seeds 0,1,2
3 runs: groups=['a5'] arms=['rlt_nope'] seeds=[0, 1, 2]
[1/3] a5_rlt_nope_s0  (0s elapsed)



In [ ]:
# sweep.py exited but stayed a zombie: this kernel is its parent and never called
# wait(), and `kill -0 <pid>` returns success for a zombie, so the follow-up's
# wait loop could never end. Reap it.
print("returncode:", sweep.poll())
time.sleep(15)
print(subprocess.run(["bash","-lc","ps -p 13599 -o pid= 2>/dev/null || echo 'reaped'; "
                      "grep -vE '^\\+ (sleep|kill)' /content/work/followup.log | tail -5"],
                     capture_output=True,text=True).stdout)

returncode: 0
reaped
+ git pull -q --ff-only
+ P=/usr/bin/python3
+ /usr/bin/python3 sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 --groups a5 --arms rlt --seeds 2,3
2 runs: groups=['a5'] arms=['rlt'] seeds=[2, 3]
[1/2] a5_rlt_s2  (0s elapsed)



In [ ]:
r = subprocess.run([sys.executable,"analyze.py","--summary"],capture_output=True,text=True)
open("results/summary.json","w").write(r.stdout.strip())
print(len(r.stdout), "chars")
print(r.stdout.strip())

2956 chars
{"a5_plain_s0":{"eval":{"16":[0.1426,0.0117],"32":[0.0802,0.0156],"64":[0.0489,0.0098],"128":[0.0316,0.0195],"256":[0.0247,0.0117]},"params":4227840,"train_acc":0.0813,"train_loss":3.8281,"ms_per_step":143,"length":32,"layers":2,"steps":1200,"rope":1},"a5_plain_s1":{"eval":{"16":[0.2244,0.0195],"32":[0.1212,0.0078],"64":[0.0684,0.0117],"128":[0.0423,0.0215],"256":[0.0297,0.0117]},"params":4227840,"train_acc":0.1223,"train_loss":3.6237,"ms_per_step":143,"length":32,"layers":2,"steps":1200,"rope":1},"a5_rlt_a0_s0":{"eval":{"16":[0.421,0.0117],"32":[0.212,0.0137],"64":[0.1181,0.0254],"128":[0.0669,0.0234],"256":[0.041,0.0117]},"params":2722048,"train_acc":0.2139,"train_loss":3.2302,"ms_per_step":289,"length":32,"layers":2,"steps":1200,"rope":1},"a5_rlt_a0_s1":{"eval":{"16":[0.1685,0.0176],"32":[0.0928,0.0078],"64":[0.0543,0.0117],"128":[0.0352,0.0215],"256":[0.0266,0.0117]},"params":2722048,"train_acc":0.0928,"train_loss":3.716,"ms_per_step":274,"length":32,"layers":2,"steps":1

In [ ]:
time.sleep(290)
print(subprocess.run(["bash","-lc","date; grep -vE '^\\+ (sleep|kill)' /content/work/followup.log | tail -8; echo; "
                      "for f in logs/a5_rlt_s2.log logs/a5_rlt_s3.log logs/a5_rlt_nope_s0.log; do echo \"== $f\"; grep 'eval ' $f 2>/dev/null; done"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 06:44:06 AM UTC 2026
+ cd /content/rlt-reproduce
+ git pull -q --ff-only
+ P=/usr/bin/python3
+ /usr/bin/python3 sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 --groups a5 --arms rlt --seeds 2,3
2 runs: groups=['a5'] arms=['rlt'] seeds=[2, 3]
[1/2] a5_rlt_s2  (0s elapsed)
[2/2] a5_rlt_s3  (335s elapsed)

== logs/a5_rlt_s2.log
  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 0.9423  final-position acc 0.8281
  eval L= 128  acc 0.6819  final-position acc 0.1426
  eval L= 256  acc 0.3698  final-position acc 0.0254
== logs/a5_rlt_s3.log
== logs/a5_rlt_nope_s0.log



In [ ]:
time.sleep(295)
print(subprocess.run(["bash","-lc","date; grep -vE '^\\+ (sleep|kill)' /content/work/followup.log | tail -6; echo; ls results/ | tr '\\n' ' '"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 06:49:12 AM UTC 2026
[1/2] a5_rlt_s2  (0s elapsed)
[2/2] a5_rlt_s3  (335s elapsed)
sweep done in 670s
+ /usr/bin/python3 sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 --groups a5 --arms rlt_nope --seeds 0,1,2
3 runs: groups=['a5'] arms=['rlt_nope'] seeds=[0, 1, 2]
[1/3] a5_rlt_nope_s0  (0s elapsed)

a5_plain_s0.json a5_plain_s1.json a5_rlt_a0_s0.json a5_rlt_a0_s1.json a5_rlt_s0.json a5_rlt_s1.json a5_rlt_s2.json a5_rlt_s3.json summary.json z60_plain_s0.json z60_plain_s1.json z60_rlt_a0_s0.json z60_rlt_a0_s1.json z60_rlt_s0.json z60_rlt_s1.json 


In [ ]:
time.sleep(295)
print(subprocess.run(["bash","-lc","date; grep -vE '^\\+ (sleep|kill|cd|git|P=)' /content/work/followup.log | tail -6; echo; "
                      "for f in logs/a5_rlt_nope_s*.log; do echo \"== $f\"; grep 'eval ' $f 2>/dev/null; done"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 06:59:28 AM UTC 2026
sweep done in 670s
+ /usr/bin/python3 sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 --groups a5 --arms rlt_nope --seeds 0,1,2
3 runs: groups=['a5'] arms=['rlt_nope'] seeds=[0, 1, 2]
[1/3] a5_rlt_nope_s0  (0s elapsed)
[2/3] a5_rlt_nope_s1  (343s elapsed)
[3/3] a5_rlt_nope_s2  (661s elapsed)

== logs/a5_rlt_nope_s0.log
  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 1.0000  final-position acc 1.0000
  eval L= 128  acc 0.9999  final-position acc 0.9980
  eval L= 256  acc 0.9931  final-position acc 0.9688
== logs/a5_rlt_nope_s1.log
  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 1.0000  final-position acc 1.0000
  eval L= 128  acc 1.0000  final-position acc 1.0000
  eval L= 256  acc 0.9980  final-position acc 0.9922
== logs/a5_rlt_nope_s2.log



In [ ]:
time.sleep(295)
print(subprocess.run(["bash","-lc","date; grep -vE '^\\+ (sleep|kill|cd|git|P=)' /content/work/followup.log | tail -8; echo; "
                      "for f in logs/a5_rlt_nope_s2.log logs/a5_rlt_untied_s0.log; do echo \"== $f\"; grep 'eval ' $f 2>/dev/null; done"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 07:04:36 AM UTC 2026
3 runs: groups=['a5'] arms=['rlt_nope'] seeds=[0, 1, 2]
[1/3] a5_rlt_nope_s0  (0s elapsed)
[2/3] a5_rlt_nope_s1  (343s elapsed)
[3/3] a5_rlt_nope_s2  (661s elapsed)
sweep done in 979s
+ /usr/bin/python3 sweep.py --steps 1200 --length 32 --batch 384 --layers 2 --d-model 256 --groups a5 --arms rlt_untied --seeds 0
1 runs: groups=['a5'] arms=['rlt_untied'] seeds=[0]
[1/1] a5_rlt_untied_s0  (0s elapsed)

== logs/a5_rlt_nope_s2.log
  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 1.0000  final-position acc 1.0000
  eval L= 128  acc 0.9978  final-position acc 0.9863
  eval L= 256  acc 0.9582  final-position acc 0.8223
== logs/a5_rlt_untied_s0.log



In [ ]:
time.sleep(295)
print(subprocess.run(["bash","-lc","date; grep -E 'FOLLOWUP_DONE|VERDICT|sweep done' /content/work/followup.log | tail -5; echo; "
                      "grep 'eval ' logs/a5_rlt_untied_s0.log 2>/dev/null"],
                     capture_output=True,text=True).stdout)

Sun Sep 13 07:09:44 AM UTC 2026
sweep done in 670s
sweep done in 979s
sweep done in 336s
+ echo FOLLOWUP_DONE
FOLLOWUP_DONE

  eval L=  16  acc 1.0000  final-position acc 1.0000
  eval L=  32  acc 1.0000  final-position acc 1.0000
  eval L=  64  acc 0.9985  final-position acc 0.9941
  eval L= 128  acc 0.9958  final-position acc 0.9824
  eval L= 256  acc 0.9724  final-position acc 0.8906



In [ ]:
print(subprocess.run(["bash","-lc","tail -60 /content/work/followup.log | grep -vE '^\\+ '"],
                     capture_output=True,text=True).stdout)

2 runs: groups=['a5'] arms=['rlt'] seeds=[2, 3]
[1/2] a5_rlt_s2  (0s elapsed)
[2/2] a5_rlt_s3  (335s elapsed)
sweep done in 670s
3 runs: groups=['a5'] arms=['rlt_nope'] seeds=[0, 1, 2]
[1/3] a5_rlt_nope_s0  (0s elapsed)
[2/3] a5_rlt_nope_s1  (343s elapsed)
[3/3] a5_rlt_nope_s2  (661s elapsed)
sweep done in 979s
1 runs: groups=['a5'] arms=['rlt_untied'] seeds=[0]
[1/1] a5_rlt_untied_s0  (0s elapsed)
sweep done in 336s
C6  Sec. 5.3 / App. A.4 -- current-policy replay
  batch=128 prompt_len=16 n_new=2 reward mean=0.0312 (chance = 0.0167)

  (a) Replay at the SAME parameters that produced the rollout.
      Sec. 5.3 predicts r_i = 1 exactly for the reference schedule.

      schedule       mean r    max |r-1|   max |log r|
      exact        1.000000    0.000e+00     0.000e+00
      nograd       1.000000    0.000e+00     0.000e+00
      stale        1.000000    0.000e+00     0.000e+00
      reset        0.976021    1.910e+00     1.541e+00

  (b) Replay AFTER one optimizer step, i.e. the or

In [ ]:
sh("git","pull","-q","--ff-only")
sh(sys.executable,"rl.py","--rl-steps","0","--out","results/c6_replay.json")


C6  Sec. 5.3 / App. A.4 -- current-policy replay
  batch=128 prompt_len=16 n_new=2 reward mean=0.0312 (chance = 0.0167)

  (a) Replay at the SAME parameters that produced the rollout.
      Sec. 5.3 predicts r_i = 1 exactly for the reference schedule.

      schedule       mean r    max |r-1|   max |log r|
      exact        1.000000    0.000e+00     0.000e+00
      nograd       1.000000    0.000e+00     0.000e+00
      stale        1.000000    0.000e+00     0.000e+00
      reset        0.976021    1.910e+00     1.541e+00

  (b) Replay after N optimizer steps -- the ordinary training case.
      Staleness is a POST-UPDATE phenomenon. Sec. 5.4 places it precisely:
      'After an optimizer update, previously computed encoder KV, recurrent
      outputs, and decoder SWA KV generally cease to be current-policy
      values.'  So the error should be absent at N = 0 and grow from N = 1.

      updates        nograd         stale         reset
               max |log p -  max |log p -  max 

0

In [ ]:
sh("git","pull","-q","--ff-only")
r = subprocess.run([sys.executable,"analyze.py","--summary"],capture_output=True,text=True)
if r.returncode: print(r.stderr[-2000:])
open("results/summary.json","w").write(r.stdout.strip())
import json as _j
d = _j.loads(r.stdout)
print(len(d), "runs\n")
NEW = [k for k in d if "nope" in k or "untied" in k or k.endswith(("_s2","_s3"))]
print(_j.dumps({k: d[k]["eval"] for k in NEW}, separators=(",",":")))
print()
sh(sys.executable,"analyze.py")


18 runs

{"a5_rlt_nope_s0":{"16":[1.0,1.0],"32":[1.0,1.0],"64":[1.0,1.0],"128":[0.9999,0.998],"256":[0.9931,0.9688]},"a5_rlt_nope_s1":{"16":[1.0,1.0],"32":[1.0,1.0],"64":[1.0,1.0],"128":[1.0,1.0],"256":[0.998,0.9922]},"a5_rlt_nope_s2":{"16":[1.0,1.0],"32":[1.0,1.0],"64":[1.0,1.0],"128":[0.9978,0.9863],"256":[0.9582,0.8223]},"a5_rlt_s2":{"16":[1.0,1.0],"32":[1.0,1.0],"64":[0.9423,0.8281],"128":[0.6819,0.1426],"256":[0.3698,0.0254]},"a5_rlt_s3":{"16":[1.0,1.0],"32":[1.0,1.0],"64":[0.9852,0.9414],"128":[0.7939,0.334],"256":[0.4872,0.0625]},"a5_rlt_untied_s0":{"16":[1.0,1.0],"32":[1.0,1.0],"64":[0.9985,0.9941],"128":[0.9958,0.9824],"256":[0.9724,0.8906]}}

Accuracy on prefix products.  chance = 0.0167, ceiling = 1.0000, trained at length 32
mean +/- std over seeds; * marks the training length

--- A5  (non-abelian, NC1-complete word problem) ---
  arm                              seeds        L=16         L=32*        L=64        L=128        L=256 
  RLT (alpha=1, tied)                  

0

In [ ]:
sh("git","pull","-q","--ff-only")
ok = True
for cmd in (["check_task.py"], ["smoke.py","--device","cuda"], ["smoke_static.py","--length","32"]):
    rc = subprocess.run([sys.executable,*cmd],capture_output=True,text=True)
    tail = [l for l in rc.stdout.strip().split("\n") if "VERDICT" in l or "GATE" in l]
    print(f"{' '.join(cmd):<32s} rc={rc.returncode}")
    for t in tail: print("   ", t.strip())
    if rc.returncode: ok = False; print(rc.stderr[-1500:])
print("\nALL GREEN" if ok else "\nFAILURES PRESENT")


check_task.py                    rc=0
    GATE: PASS -- best degenerate/shortcut policy scores 0.0322 against a 0.0167 floor.
smoke.py --device cuda           rc=0
    VERDICT: CONFIRMED (split difference is 2.2x the noise floor)
    VERDICT: CONFIRMED (bitwise)
    VERDICT: CONFIRMED
    VERDICT: CONFIRMED
    VERDICT: CONFIRMED (path exists and carries gradient)
smoke_static.py --length 32      rc=0
    VERDICT: CONFIRMED

ALL GREEN


In [ ]:
print(open("results/c6_replay.json").read())

{
  "exact": {
    "mean_r": 1.0,
    "max_abs_r_minus_1": 0.0,
    "max_abs_logr": 0.0
  },
  "nograd": {
    "mean_r": 1.0,
    "max_abs_r_minus_1": 0.0,
    "max_abs_logr": 0.0
  },
  "stale": {
    "mean_r": 1.0,
    "max_abs_r_minus_1": 0.0,
    "max_abs_logr": 0.0
  },
  "reset": {
    "mean_r": 0.976021409034729,
    "max_abs_r_minus_1": 1.909632682800293,
    "max_abs_logr": 1.5405795574188232
  },
  "drift": {
    "nograd": {
      "1": 0.0,
      "2": 0.0,
      "5": 0.0,
      "10": 0.0,
      "20": 0.0
    },
    "stale": {
      "1": 0.01596212387084961,
      "2": 0.03193807601928711,
      "5": 0.0798649787902832,
      "10": 0.15921354293823242,
      "20": 0.31232404708862305
    },
    "reset": {
      "1": 1.533508539199829,
      "2": 1.526402473449707,
      "5": 1.5048916339874268,
      "10": 1.4821314811706543,
      "20": 1.465139389038086
    }
  },
  "grad": {
    "cos": 0.31664514541625977,
    "norm_exact": 0.6018099188804626,
    "norm_nograd": 0.186592489